# ActionShap — complete end-to-end runner

This is the canonical **Run All** notebook for revision 4. It performs the entire reproducible workflow from an empty local data directory:

1. resolves the repository and installs the locked recommendation environment;
2. downloads MovieLens-1M and Amazon Digital Music from their declared sources;
3. verifies/prepares content-addressed datasets;
4. loads and audits both temporal splits;
5. inspects the frozen real-data preflight;
6. runs all independent convergence studies;
7. runs the two-dataset, two-model, five-seed primary/full-catalogue matrix;
8. runs every predeclared sensitivity;
9. creates hierarchical statistics, tables, figures, manifests, and the result archive;
10. validates the manuscript and displays final status.

**Expected cost:** the final suite is intentionally expensive and may take hours. Raw datasets, environments, and raw JSON are ignored by Git. The notebook never turns a failed gate or incomplete matrix into a paper claim.


In [16]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

# Works when opened from the repository root or from ActionShap/code.
START = Path.cwd().resolve()
CANDIDATES = [
    START,
    START / "paper-ideas" / "ActionShap" / "code",
]
CODE_ROOT = next(
    (candidate for candidate in CANDIDATES if (candidate / "scripts" / "run_final_suite.py").exists()),
    None,
)
if CODE_ROOT is None:
    raise RuntimeError(
        "Cannot locate paper-ideas/ActionShap/code. Open the notebook from the "
        "repository root or from its code directory."
    )
ACTIONSHAP_ROOT = CODE_ROOT.parent
REPO_ROOT = ACTIONSHAP_ROOT.parent.parent
CONFIG_PATH = CODE_ROOT / "configs" / "final.yaml"
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# Run-All controls. The defaults execute the full requested workflow.
INSTALL_DEPENDENCIES = True
DOWNLOAD_DATA_IF_MISSING = True
ACCEPT_DATASET_TERMS = True  # Set only after reviewing GroupLens and Amazon terms/citations.
FORCE_REDOWNLOAD = False
RUN_FINAL_SUITE = True

print("REPO_ROOT       =", REPO_ROOT)
print("CODE_ROOT       =", CODE_ROOT)
print("CONFIG_PATH     =", CONFIG_PATH)
print("Python          =", sys.version.split()[0])


REPO_ROOT       = /Users/mlouhichi/Desktop/temp/next-paper
CODE_ROOT       = /Users/mlouhichi/Desktop/temp/next-paper/paper-ideas/ActionShap/code
CONFIG_PATH     = /Users/mlouhichi/Desktop/temp/next-paper/paper-ideas/ActionShap/code/configs/final.yaml
Python          = 3.12.13


## 1. Install the locked environment

The lock file is the exact tested Python environment. Installation is idempotent. If a platform cannot resolve a locked wheel, set `INSTALL_DEPENDENCIES=False`, install `requirements-recommendation.txt` manually, and preserve the resolved versions in result provenance.


In [17]:
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(CODE_ROOT / "requirements-recommendation.lock"),
        ],
        cwd=CODE_ROOT,
        check=True,
    )
else:
    print("Dependency installation skipped by configuration.")


  Using cached matplotlib-3.11.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (80 kB)
  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (11 kB)
Using cached matplotlib-3.11.1-cp312-cp312-macosx_11_0_arm64.whl (9.3 MB)
Using cached narwhals-2.24.0-py3-none-any.whl (461 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 7.6 MB/s  0:00:01m0:00:010:01
Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl (173 kB)
Using cached scikit_learn-1.9.0-cp312-cp312-macosx_12_0_arm64.whl (8.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 6.0 MB/s  0:00:03 eta 0:00:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.18.0
    Uninstalling scipy-1.18.0:
      Successfully uninstalled scipy-1.18.0
  Attempting uninstall: PyYAML━━━━━━━━━━━━━━━━━━ 0/7 [scipy]
    Found existing installa

In [18]:
# Import only after dependency installation.
import numpy as np
import pandas as pd
import scipy
import sklearn
import yaml

from actionshap.recommendation_data import (
    load_interactions_csv,
    load_movielens_1m,
)

CONFIG = yaml.safe_load(CONFIG_PATH.read_text())
print(
    {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit-learn": sklearn.__version__,
    }
)
print(yaml.safe_dump(CONFIG, sort_keys=False))


{'numpy': '2.4.6', 'pandas': '2.2.2', 'scipy': '1.18.0', 'scikit-learn': '1.9.0'}
datasets:
- name: MovieLens-1M
  format: ml1m
  path: data/ml-1m/ratings.dat
  rating_threshold: 4.0
- name: Amazon-Digital-Music
  format: csv
  path: data/amazon-digital-music/interactions.csv
  user_column: user
  item_column: item
  timestamp_column: timestamp
  rating_column: rating
  rating_threshold: 4.0
primary_model: itemknn
models:
- itemknn
- profile
seeds:
- 42
- 43
- 44
- 45
- 46
candidate_seed: 1729
user_seed: 2718
tie_seed: 31415
max_users: 1000
full_catalog_users: 250
robustness_null_draws: 100
minimum_interactions: 4
n_max: 20
evaluation_size: 200
k: 10
utility: target_margin
action_rho: 0.5
budget: 2
permutations: 500
lime_samples: 512
null_draws: 1000
gate_users: 200
oracle_users: 0
epochs: 10
embedding_dim: 64
profile_samples_per_user: 1
profile_learning_rate: 0.03
profile_regularization: 0.0001
itemknn_neighbours: 200
convergence_users: 100
convergence_budgets:
- 25
- 50
- 100
- 250
-

## 2. Download and prepare both datasets

Sources are declared in `scripts/download_datasets.py`:

- GroupLens MovieLens-1M; the extracted `ratings.dat` is checked against the pinned SHA-256 used by the tracked preflight.
- Amazon Review Data (2018), `Digital_Music_5.json.gz`; the builder retains ratings at least 4, resolves duplicates deterministically, reapplies iterative 5-core filtering, and writes source/output hashes.

The downloader requires explicit terms acceptance and uses atomic `.part` files. If institutional networking blocks downloads, place the unmodified source files at the documented locations and rerun this cell.


In [19]:
if DOWNLOAD_DATA_IF_MISSING:
    if not ACCEPT_DATASET_TERMS:
        raise RuntimeError(
            "Review the GroupLens and Amazon dataset terms, then set "
            "ACCEPT_DATASET_TERMS=True."
        )
    command = [
        sys.executable,
        str(CODE_ROOT / "scripts" / "download_datasets.py"),
        "--dataset",
        "all",
        "--accept-dataset-terms",
    ]
    if FORCE_REDOWNLOAD:
        command.append("--force")
    subprocess.run(command, cwd=CODE_ROOT, check=True)
else:
    print("Dataset download skipped; existing files will be audited next.")


Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_4/Frameworks/Python.framework/Versions/3.12/lib/python3.12/urllib/request.py", line 1344, in do_open
    h.request(req.get_method(), req.selector, req.data, headers,
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_4/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1358, in request
    self._send_request(method, url, body, headers, encode_chunked)
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_4/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1404, in _send_request
    self.endheaders(body, encode_chunked=encode_chunked)
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_4/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1353, in endheaders
    self._send_output(message_body, encode_chunked=encode_chunked)
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_4/Frameworks/Python.framework/Versions/3.12/lib/python3.12/h

CalledProcessError: Command '['/Users/mlouhichi/Desktop/temp/.venv/bin/python', '/Users/mlouhichi/Desktop/temp/next-paper/paper-ideas/ActionShap/code/scripts/download_datasets.py', '--dataset', 'all', '--accept-dataset-terms']' returned non-zero exit status 1.

## 3. Load and audit temporal data

This cell proves that both prepared files are readable under the exact final configuration. It reports post-filter users/items/interactions, history lengths, hashes, and deterministic split reconstruction. It does **not** inspect explanation outcomes.


In [20]:
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_configured_dataset(spec):
    source = (CODE_ROOT / spec["path"]).resolve()
    if not source.exists():
        raise FileNotFoundError(source)
    if spec["format"] == "ml1m":
        data = load_movielens_1m(
            source,
            rating_threshold=float(spec.get("rating_threshold", 4.0)),
            minimum_interactions=int(CONFIG["minimum_interactions"]),
        )
        repeat = load_movielens_1m(
            source,
            rating_threshold=float(spec.get("rating_threshold", 4.0)),
            minimum_interactions=int(CONFIG["minimum_interactions"]),
        )
    else:
        kwargs = {
            "user_column": spec.get("user_column", "user"),
            "item_column": spec.get("item_column", "item"),
            "timestamp_column": spec.get("timestamp_column", "timestamp"),
            "rating_column": spec.get("rating_column", "rating"),
            "rating_threshold": float(spec.get("rating_threshold", 4.0)),
            "minimum_interactions": int(CONFIG["minimum_interactions"]),
        }
        data = load_interactions_csv(source, **kwargs)
        repeat = load_interactions_csv(source, **kwargs)
    if data.test != repeat.test or data.validation != repeat.validation:
        raise AssertionError(f"Temporal split is not deterministic for {spec['name']}")
    lengths = np.array([len(history) for history in data.train.values()])
    return data, {
        "dataset": spec["name"],
        "path": str(source.relative_to(REPO_ROOT)),
        "sha256": file_sha256(source),
        "users": len(data.test),
        "items": data.n_items,
        "interactions_after_filter": int(lengths.sum() + 2 * len(lengths)),
        "train_history_median": float(np.median(lengths)),
        "train_history_p95": float(np.quantile(lengths, 0.95)),
        "minimum_train_history": int(lengths.min()),
        "deterministic_split": True,
    }

LOADED_DATASETS = {}
audit_rows = []
for dataset_spec in CONFIG["datasets"]:
    data, audit = load_configured_dataset(dataset_spec)
    LOADED_DATASETS[dataset_spec["name"]] = data
    audit_rows.append(audit)

DATA_AUDIT = pd.DataFrame(audit_rows)
display(DATA_AUDIT)
assert len(LOADED_DATASETS) == 2
assert all(row["minimum_train_history"] >= 2 for row in audit_rows)


FileNotFoundError: /Users/mlouhichi/Desktop/temp/next-paper/paper-ideas/ActionShap/code/data/amazon-digital-music/interactions.csv

## 4. Inspect the frozen design preflight

The preflight selected primary ItemKNN, `n_max=20`, target-margin attribution, and the conservative final floor `M=500` before the final result matrix. It is content-addressed and excluded from headline result aggregation.


In [ ]:
preflight_path = ACTIONSHAP_ROOT / "paper" / "preflight" / "movielens_masking_gate.json"
PREFLIGHT = json.loads(preflight_path.read_text())
movielens_hash = DATA_AUDIT.loc[
    DATA_AUDIT["dataset"] == "MovieLens-1M", "sha256"
].iloc[0]
if PREFLIGHT["source_sha256"] != movielens_hash:
    raise AssertionError("MovieLens payload differs from the frozen design preflight")

gate_frame = pd.DataFrame(PREFLIGHT["final_window_gate_runs"])
display(gate_frame)
print(PREFLIGHT["decision"])
print(PREFLIGHT["utility_decision"])
assert gate_frame.loc[gate_frame["model"] == "itemknn", "passed"].all()
assert PREFLIGHT["utility_convergence_preflight"]["target_margin"][
    "selected_permutations"
] == 250


## 5. Execute the complete final suite

The suite runs **85 scientific commands** plus asset/manuscript/package validation. It stops on any blocking primary gate, missing experiment, inconsistent cohort, inadequate target-margin convergence, or primary-quality failure. Robustness-model and NDCG-utility failures remain visible as bounded findings.

Set `RUN_FINAL_SUITE=False` only when you want to audit already-generated schema-v2 raw files without rerunning models.


In [ ]:
if RUN_FINAL_SUITE:
    subprocess.run(
        [
            sys.executable,
            str(CODE_ROOT / "scripts" / "run_final_suite.py"),
            "--config",
            str(CONFIG_PATH),
        ],
        cwd=CODE_ROOT,
        check=True,
    )
else:
    print("Final suite execution skipped; validating existing outputs only.")
    subprocess.run(
        [sys.executable, str(CODE_ROOT / "scripts" / "make_paper_assets.py")],
        cwd=CODE_ROOT,
        check=True,
    )


## 6. Inspect final validation and headline assets

Only `status: PASS` permits numerical claims. The manuscript intentionally keeps result placeholders until the final tables exist and authors write conclusions that match the validated data.


In [ ]:
validation_path = ACTIONSHAP_ROOT / "paper" / "final" / "manifests" / "validation_report.json"
manifest_path = ACTIONSHAP_ROOT / "paper" / "final" / "manifests" / "asset_manifest.json"
VALIDATION = json.loads(validation_path.read_text())
print(json.dumps(VALIDATION, indent=2))
if VALIDATION["status"] != "PASS":
    raise RuntimeError("Final ActionShap matrix is incomplete or invalid; inspect the report above.")

MANIFEST = json.loads(manifest_path.read_text())
print("Source files:", len(MANIFEST["source_files"]))
print("Generated files:", len(MANIFEST["generated_files"]))

method_table = ACTIONSHAP_ROOT / "paper" / "final" / "tables" / "method_metrics.csv"
paired_table = ACTIONSHAP_ROOT / "paper" / "final" / "tables" / "paired_tests.csv"
display(pd.read_csv(method_table).head(30))
display(pd.read_csv(paired_table).head(30))


## 7. Validate manuscript and release package

Static validation checks citations, environments, pilot-value contamination, generated tables, final asset status, and remaining placeholders. `--require-final` is intentionally not used here until authors replace the result prose placeholders.


In [ ]:
subprocess.run(
    [sys.executable, str(CODE_ROOT / "scripts" / "validate_manuscript.py")],
    cwd=CODE_ROOT,
    check=True,
)

release_root = CODE_ROOT / "results" / "release"
archives = sorted(release_root.glob("actionshap-schema-v2-results.tar.gz*"))
print("Release artifacts:")
for artifact in archives:
    print(" -", artifact, artifact.stat().st_size, "bytes")


## Completion checklist

A successful Run All ends with:

- two audited datasets and matching SHA-256 provenance;
- primary ItemKNN gates passing for every dataset/seed;
- target-margin convergence and an explicitly reported NDCG stress test;
- exact dual-utility `B<=2` oracles for all primary users;
- five methods on matched cohorts across all required conditions;
- `paper/final/manifests/validation_report.json` equal to `PASS`;
- generated tables/figures and a content-addressed raw-result archive;
- manuscript static validation with only result-writing placeholders remaining.

After writing the numerical Results, Discussion, Abstract, and Conclusion, run:

```bash
python scripts/validate_manuscript.py --require-final
```
